# Intent labeling — MongoDB + Qwen2.5 + guardrail

Pipeline: **truy hồi ứng viên từ MongoDB** → **Qwen sinh JSON** → **kiểm taxonomy + ngưỡng** → lưu `intent_annotations`.

Xem `LABELING_GUIDE.md` (auto-add nhãn mới mặc định **confidence ≥ 0.96**).

**Điều kiện:** graph đã có trong MongoDB (`intent_nodes` / `intent_edges`) — build từ `data/intent_graph_rag_colab.ipynb`.


## 1. Cài đặt


In [1]:
import sys
print(sys.executable)


/venv/main/bin/python


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
pip install -q pymongo certifi transformers accelerate torch sentencepiece protobuf sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình


In [ ]:
import os
import getpass
from pathlib import Path

REPO_ROOT = Path(os.environ.get("INTENT_REPO", ".")).resolve()
HASAKI_JSON = REPO_ROOT / "data/hasaki_prelabel.json"

DB_NAME = os.environ.get("MONGODB_DB", "intent_kb")
COL_NODES = "intent_nodes"
COL_EDGES = "intent_edges"
# Khong hardcode URI: uu tien bien moi truong MONGODB_URI, neu trong thi nhap khi chay cell (an khi go)
MONGODB_URI = (os.environ.get("MONGODB_URI") or "").strip()
if not MONGODB_URI:
    MONGODB_URI = getpass.getpass("MONGODB_URI (dan vao roi Enter): ").strip()

MODEL_ID = os.environ.get("QWEN_MODEL", "Qwen/Qwen2.5-7B-Instruct")

MIN_CONF_APPROVE_EXISTING = 0.90
MIN_CONF_AUTO_ADD_NEW_LABEL = 0.96
ALLOW_AUTO_ADD_NEW_LABEL = True

BATCH_START = 0
HASAKI_BATCH_SIZE = 200
# Resume chay full prelabel: index bat dau (0 = tu dau file)
HASAKI_RUN_OFFSET = 0
# Sau khi export JSON: xoa BATCH_OUTPUT khoi RAM (checkpoint van tren dia)
HASAKI_FREE_MEM_AFTER_EXPORT = True
BATCH_LIMIT = HASAKI_BATCH_SIZE
TOP_K_CANDIDATES = 12
MAX_NEW_TOKENS = 384

print("HASAKI_JSON:", HASAKI_JSON.exists(), HASAKI_JSON)
print("MONGODB_URI set:", bool(MONGODB_URI))


HASAKI_JSON: True /workspace/data/hasaki_prelabel.json
MONGODB_URI set: True


## 3. Kết nối MongoDB


In [5]:
import certifi
from pymongo import MongoClient

if not MONGODB_URI:
    raise ValueError(
        "Chua co MONGODB_URI. Dat bien moi truong MONGODB_URI hoac chay lai cell cau hinh de nhap."
    )

client = MongoClient(
    MONGODB_URI,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=15000,
)
client.admin.command("ping")
db = client[DB_NAME]
print("OK db:", db.name)
print("intent_nodes:", db[COL_NODES].count_documents({}))
print("intent_edges:", db[COL_EDGES].count_documents({}))


OK db: intent_kb
intent_nodes: 181
intent_edges: 182


## 4. Helper: upsert graph, guardrail, retrieval


In [ ]:
from datetime import datetime, timezone
import json
import re
import unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ngưỡng snap nhãn intent có sẵn qua cosine vs embedding đã enrich (đối xứng với retrieval)
SEMANTIC_SNAP_TO_INTENT_THRESHOLD = 0.85


# Global semantic encoder - lazy loaded
_semantic_model = None

def get_semantic_model():
    """Lazy load sentence transformer model"""
    global _semantic_model
    if _semantic_model is None:
        print("Loading paraphrase-multilingual-MiniLM-L12-v2...")
        _semantic_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _semantic_model


def build_graph_aware_intent_text(intent_doc, db):
    """Build graph-aware description for intent node embedding
    
    Args:
        intent_doc: intent node document from MongoDB
        db: MongoDB database connection
    
    Returns:
        str: graph-aware text for embedding
    """
    # Extract path components
    l1 = intent_doc.get('l1', '')
    l2 = intent_doc.get('l2', '') 
    l3 = intent_doc.get('l3', '')
    description = intent_doc.get('description', '')
    detection_signals = intent_doc.get('detection_signals', '')
    example = intent_doc.get('example', '')
    
    # Build full path
    full_path = f"{l1}.{l2}.{l3}"
    
    # Get sibling intents (same L2, limit 5)
    siblings_cursor = db[COL_NODES].find(
        {'level': 'intent', 'l2': l2, '_id': {'$ne': intent_doc['_id']}}, 
        {'l3': 1}
    ).limit(5)
    siblings = [doc.get('l3', '') for doc in siblings_cursor if doc.get('l3')]
    siblings_text = ', '.join(siblings) if siblings else 'None'
    
    # Construct graph-aware text
    graph_text = f"""Intent: {full_path}.
Parent: {l2}.
Siblings: {siblings_text}.
Description: {description}
Signals: {detection_signals}
Example: {example}"""
    
    return graph_text


def embed_and_store_intent_nodes(db):
    """One-time embedding enrichment for all intent nodes
    
    Builds graph-aware text and stores embeddings in MongoDB
    """
    model = get_semantic_model()
    col_nodes = db[COL_NODES]
    
    # Find all intent nodes without embeddings
    intent_nodes = list(col_nodes.find({'level': 'intent'}))
    print(f"Found {len(intent_nodes)} intent nodes to process")
    
    updated_count = 0
    for intent_doc in intent_nodes:
        # Skip if already has embedding
        if 'embedding' in intent_doc and 'graph_aware_text' in intent_doc:
            continue
            
        # Build graph-aware text
        graph_text = build_graph_aware_intent_text(intent_doc, db)
        
        # Generate embedding
        embedding = model.encode([graph_text])[0]
        # Normalize embedding
        embedding = embedding / np.linalg.norm(embedding)
        
        # Store in MongoDB
        col_nodes.update_one(
            {'_id': intent_doc['_id']},
            {
                '$set': {
                    'graph_aware_text': graph_text,
                    'embedding': embedding.tolist(),  # Convert to list for MongoDB
                    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
                    'embedding_updated_at': datetime.now(timezone.utc)
                }
            }
        )
        updated_count += 1
        
        if updated_count % 10 == 0:
            print(f"Processed {updated_count} nodes...")
    
    print(f"✅ Embedded {updated_count} intent nodes")
    return updated_count


def semantic_retrieval(db, query_text, top_k=10, min_similarity=0.3):
    """Semantic retrieval using graph-aware embeddings
    
    Args:
        db: MongoDB database connection
        query_text: user query sentence
        top_k: number of candidates to return
        min_similarity: minimum cosine similarity threshold
    
    Returns:
        list: intent documents with similarity scores
    """
    model = get_semantic_model()
    col_nodes = db[COL_NODES]
    
    # Encode query
    query_embedding = model.encode([query_text])[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    
    # Find intent nodes with embeddings
    intent_nodes = list(col_nodes.find(
        {'level': 'intent', 'embedding': {'$exists': True}},
        {
            '_id': 1, 'name': 1, 'l1': 1, 'l2': 1, 'l3': 1,
            'description': 1, 'detection_signals': 1, 'example': 1,
            'product_category': 1, 'embedding': 1
        }
    ))
    
    if not intent_nodes:
        print("Warning: No intent nodes with embeddings found")
        return []
    
    # Calculate similarities
    scored_candidates = []
    for intent_doc in intent_nodes:
        stored_embedding = np.array(intent_doc['embedding'])
        
        # Compute cosine similarity
        similarity = cosine_similarity([query_embedding], [stored_embedding])[0][0]
        
        if similarity >= min_similarity:
            # Add similarity score to document
            intent_doc['semantic_score'] = float(similarity)
            scored_candidates.append(intent_doc)
    
    # Sort by similarity score descending
    scored_candidates.sort(key=lambda x: x['semantic_score'], reverse=True)
    
    return scored_candidates[:top_k]


def union_retrieval(db, text, category=None, top_k_regex=8, top_k_semantic=6, sample=None):
    """UNION of regex and semantic retrieval with deduplication
    
    Args:
        db: MongoDB database connection
        text: query text
        category: product category (optional)
        top_k_regex: max regex candidates
        top_k_semantic: max semantic candidates
        sample: sample data for context matching
    
    Returns:
        list: deduplicated union of candidates
    """
    # Get regex candidates (existing logic)
    regex_candidates = get_candidate_l3_from_mongodb(db, text, category, top_k_regex, sample)
    
    # Get semantic candidates (new logic)
    semantic_candidates = semantic_retrieval(db, text, top_k_semantic)
    
    # Deduplicate by _id, preserving document structure
    seen_ids = set()
    union_candidates = []
    
    # Add regex candidates first (preserve original scoring)
    for candidate in regex_candidates:
        if candidate['_id'] not in seen_ids:
            candidate['retrieval_method'] = 'regex'
            union_candidates.append(candidate)
            seen_ids.add(candidate['_id'])
    
    # Add semantic candidates
    for candidate in semantic_candidates:
        if candidate['_id'] not in seen_ids:
            candidate['retrieval_method'] = 'semantic'
            union_candidates.append(candidate)
            seen_ids.add(candidate['_id'])
    
    # Limit total candidates
    max_total = max(top_k_regex + top_k_semantic, TOP_K_CANDIDATES)
    return union_candidates[:max_total]


def find_nearest_existing_intent(db, pred_text, threshold=SEMANTIC_SNAP_TO_INTENT_THRESHOLD):
    """Embed prediction path/slug; cosine vs intent node embeddings. Returns (node, sim) or (None, best_sim)."""
    text = (pred_text or "").strip()
    if not text:
        return None, -1.0
    model = get_semantic_model()
    q = np.asarray(model.encode([text])[0], dtype=np.float64)
    n = np.linalg.norm(q)
    if n < 1e-9:
        return None, -1.0
    q = q / n
    nodes = list(
        db[COL_NODES].find(
            {"level": "intent", "embedding": {"$exists": True}},
            {"_id": 1, "l1": 1, "l2": 1, "l3": 1, "embedding": 1},
        )
    )
    best, best_sim = None, -1.0
    for intent_doc in nodes:
        ev = np.asarray(intent_doc["embedding"], dtype=np.float64)
        en = np.linalg.norm(ev)
        if en < 1e-9:
            continue
        sim = float(np.dot(q, ev / en))
        if sim > best_sim:
            best_sim, best = sim, intent_doc
    if best is not None and best_sim >= threshold:
        return best, best_sim
    return None, best_sim


def upsert_new_label_to_graph(
    db,
    *,
    domain,
    l1,
    l2,
    l3,
    intent_id,
    intent_name,
    description="",
    confidence="",
    product_category="",
    detection_signals="",
    example="",
    category_logic="",
):
    col_nodes = db[COL_NODES]
    col_edges = db[COL_EDGES]
    domain_id = f"domain:{domain}"
    l1_id, l2_id, l3_id = f"l1:{l1}", f"l2:{l2}", f"l3:{l3}"
    now = datetime.now(timezone.utc)

    for node_id, level, name, extra in [
        ("ROOT", "root", "Intent Knowledge Base", {}),
        (domain_id, "domain", domain, {}),
        (l1_id, "l1", l1, {}),
        (l2_id, "l2", l2, {}),
        (l3_id, "l3", l3, {}),
        (
            intent_id,
            "intent",
            intent_name,
            {
                "description": description,
                "confidence": confidence,
                "domain": domain,
                "product_category": product_category,
                "l1": l1,
                "l2": l2,
                "l3": l3,
                "detection_signals": detection_signals,
                "example": example,
                "category_logic": category_logic,
            },
        ),
    ]:
        col_nodes.update_one(
            {"_id": node_id},
            {"$setOnInsert": {"_id": node_id, "level": level}, "$set": {"name": name, **extra, "updated_at": now}},
            upsert=True,
        )

    for src, dst in [
        ("ROOT", domain_id),
        (domain_id, l1_id),
        (l1_id, l2_id),
        (l2_id, l3_id),
        (l3_id, intent_id),
    ]:
        eid = f"{src}->{dst}"
        col_edges.update_one(
            {"_id": eid},
            {"$setOnInsert": {"_id": eid, "from": src, "to": dst}, "$set": {"updated_at": now}},
            upsert=True,
        )
    return {"intent_id": intent_id, "status": "upserted"}


def _norm_label(s):
    s = unicodedata.normalize("NFC", (s or "").strip())
    return re.sub(r"\s+", " ", s).lower()


def _collect_allowed_taxonomy(db):
    l1_set, l2_set, l3_set = set(), set(), set()
    for doc in db[COL_NODES].find({"level": "l1"}, {"name": 1}):
        if doc.get("name"):
            l1_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l2"}, {"name": 1}):
        if doc.get("name"):
            l2_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l3"}, {"name": 1}):
        if doc.get("name"):
            l3_set.add(doc["name"])

    # norm -> chuỗi canonical trong Mongo (tranh trùng key: lấy deterministic theo sort)
    def _norm_to_canon(name_set):
        m = {}
        for name in sorted(name_set):
            k = _norm_label(name)
            if k and k not in m:
                m[k] = name
        return m

    return {
        "l1": l1_set,
        "l2": l2_set,
        "l3": l3_set,
        "norm_canon_l1": _norm_to_canon(l1_set),
        "norm_canon_l2": _norm_to_canon(l2_set),
        "norm_canon_l3": _norm_to_canon(l3_set),
    }


def _snap_pred_to_canonical_taxonomy(pred, allowed):
    """Ghi đè L1/L2/L3 bằng tên đúng trong graph nếu normalize khớp một nhãn có sẵn."""
    for lvl, key in [("l1", "L1"), ("l2", "L2"), ("l3", "L3")]:
        raw = pred.get(key)
        if raw is None:
            continue
        raw = str(raw).strip()
        if not raw:
            continue
        cmap = allowed.get(f"norm_canon_{lvl}", {})
        canon = cmap.get(_norm_label(raw))
        if canon:
            pred[key] = canon
    # Model thường trả nhầm slug domain/ngành làm L1 — map về giai đoạn taxonomy (truoc ban)
    lk = _norm_label(pred.get("L1"))
    if lk in ("my_pham", "trang_diem"):
        canon = allowed.get("norm_canon_l1", {}).get(_norm_label("truoc ban"))
        if canon:
            pred["L1"] = canon


def _is_label_allowed(pred, allowed):
    n1, n2, n3 = (
        _norm_label(pred.get("L1")),
        _norm_label(pred.get("L2")),
        _norm_label(pred.get("L3")),
    )
    nl1 = {_norm_label(x) for x in allowed["l1"]}
    nl2 = {_norm_label(x) for x in allowed["l2"]}
    nl3 = {_norm_label(x) for x in allowed["l3"]}
    return (n1 in nl1) and (n2 in nl2) and (n3 in nl3)


def is_sentence_too_ambiguous_to_annotate(sentence, min_length=6, min_meaningful_words=1):
    """Check if sentence is too short/ambiguous to reliably annotate
    
    Args:
        sentence: input sentence text
        min_length: minimum character length (excluding spaces)
        min_meaningful_words: minimum meaningful words (length >= 2)
    
    Returns:
        tuple: (is_ambiguous: bool, reason: str)
    """
    if not sentence or not sentence.strip():
        return True, "empty_sentence"
    
    text = sentence.strip()
    
    # Check character length (excluding spaces) - reduced from 8 to 6
    char_count = len(re.sub(r'\s+', '', text))
    if char_count < min_length:
        return True, f"too_short_chars_{char_count}"
    
    # Extract meaningful words (length >= 2, more lenient)
    words = re.findall(r'\w+', text.lower(), flags=re.UNICODE)
    # Reduced stop words list - keep common meaningful words
    stop_words = {'có', 'thế', 'này', 'kia', 'được', 'như'}
    meaningful_words = [w for w in words if len(w) >= 2 and w not in stop_words]
    
    # Special handling for price/money questions  
    money_words = {'tiền', 'giá', 'bao', 'nhiều', 'cost', 'price'}
    product_words = {'sản', 'phẩm', 'sp', 'hàng', 'tốt', 'xấu', 'quality'}
    
    has_money_context = any(w in ' '.join(meaningful_words) for w in money_words)
    has_product_context = any(w in ' '.join(meaningful_words) for w in product_words)
    
    # Allow price questions even if short
    if has_money_context or has_product_context:
        min_meaningful_words = 1
    
    if len(meaningful_words) < min_meaningful_words:
        return True, f"too_few_meaningful_words_{len(meaningful_words)}"
    
    # Check if sentence is just question words without ANY context
    question_only_patterns = [
        r'^(có\s*)?(không\s*)?(gì\s*\?*\s*)$',  # "Gì?"
        r'^(sao\s*\?*\s*)$',                    # "Sao"
        r'^(nào\s*\?*\s*)$',                    # "Nào?"
        r'^(đâu\s*\?*\s*)$'                     # "Đâu?"
    ]
    
    text_normalized = re.sub(r'\s+', ' ', text.lower().strip())
    for pattern in question_only_patterns:
        if re.match(pattern, text_normalized):
            return True, "question_word_only"
    
    # Check for single word + punctuation patterns (very short)
    if len(words) == 1 and len(text) <= 4:
        return True, "single_word_too_short"
    
    return False, ""


def save_annotation_with_guardrails(
    db,
    sample,
    pred,
    *,
    min_conf_auto_approve=MIN_CONF_APPROVE_EXISTING,
    min_conf_allow_new_label=MIN_CONF_AUTO_ADD_NEW_LABEL,
    allow_auto_add_new_label=ALLOW_AUTO_ADD_NEW_LABEL,
    model_name=MODEL_ID,
):
    col_ann = db["intent_annotations"]
    now = datetime.now(timezone.utc)
    
    # STEP 1: Check if sentence is too ambiguous to annotate
    sentence = sample.get("sentence", "").strip()
    is_ambiguous, ambiguous_reason = is_sentence_too_ambiguous_to_annotate(sentence)
    
    if is_ambiguous:
        # Skip annotation for ambiguous sentences
        ann_doc = {
            "sample_id": sample.get("sample_id"),
            "product_id": sample.get("product_id"),
            "sentence": sentence,
            "category": sample.get("category", ""),
            "model": model_name,
            "intent": {
                "level_1": "",
                "level_2": "",
                "level_3": [],
            },
            "confidence": 0.0,
            "reasoning": f"Câu quá ngắn/mơ hồ để annotate: {ambiguous_reason}",
            "qa_status": "skipped_ambiguous",
            "ambiguous_reason": ambiguous_reason,
            "new_label_pending_review": False,
            "graph_upserted": False,
            "semantic_snap_similarity": None,
            "semantic_snap_intent_id": None,
            "taxonomy_semantically_snapped": False,
            "source": sample.get("source", "hasaki"),
            "updated_at": now,
        }
        col_ann.update_one({"sample_id": ann_doc["sample_id"]}, {"$set": ann_doc}, upsert=True)
        return ann_doc
    
    # STEP 2: Continue with normal annotation logic
    allowed = _collect_allowed_taxonomy(db)
    # Snap chuỗi L1/L2/L3 → tên trong graph (truoc ban vs trước ban, underscore vs space, v.v.)
    _snap_pred_to_canonical_taxonomy(pred, allowed)
    confidence = float(pred.get("confidence") or 0)
    valid_existing = _is_label_allowed(pred, allowed)

    semantic_snap_similarity = None
    semantic_snap_intent_id = None
    used_semantic_snap = False

    # Nếu triple không khớp taxonomy: embed đường dẫn/slug prediction, snap vào intent gần nhất trong graph (đối xứng với embeddings đã có)
    if not valid_existing:
        parts = []
        for p in (pred.get("L1"), pred.get("L2"), pred.get("L3")):
            if p is not None and str(p).strip():
                parts.append(str(p).strip())
        slug_blob = ".".join(parts)
        l3_piece = (pred.get("L3") or "").strip()
        reasoning_snip = ((pred.get("reasoning") or "").strip())[:280]
        pred_text_for_snap = slug_blob or l3_piece or reasoning_snip
        sentence_ctx = (sample.get("sentence") or "").strip()
        cat_ctx = (sample.get("category") or "").strip()
        extra_ctx = (" ".join(x for x in (sentence_ctx, cat_ctx) if x)).strip()
        # Triple rỗng hoặc slug không đủ: dùng câu hỏi thật để embed nearest intent (khớp case pending L1/L2/L3 trống)
        if not pred_text_for_snap and extra_ctx:
            pred_text_for_snap = extra_ctx[:512]
        elif extra_ctx and extra_ctx[:80] not in pred_text_for_snap:
            pred_text_for_snap = f"{pred_text_for_snap}\n{extra_ctx}".strip()[:768]

        nearest, nearest_sim = find_nearest_existing_intent(db, pred_text_for_snap)
        semantic_snap_similarity = float(nearest_sim)

        if nearest is not None:
            semantic_snap_intent_id = nearest.get("_id")
            pred["L1"] = nearest.get("l1") or ""
            pred["L2"] = nearest.get("l2") or ""
            pred["L3"] = nearest.get("l3") or ""
            _snap_pred_to_canonical_taxonomy(pred, allowed)
            valid_existing = _is_label_allowed(pred, allowed)
            used_semantic_snap = valid_existing

    qa_status = "rejected"
    new_label_pending_review = False
    graph_upserted = False

    if valid_existing:
        # Snap semantic: không phụ thuộc ngưỡng confidence Qwen để thoát pending
        if used_semantic_snap:
            qa_status = "approved_snapped"
            new_label_pending_review = False
        else:
            qa_status = "approved" if confidence >= min_conf_auto_approve else "needs_review"
    else:
        new_label_pending_review = True
        qa_status = "pending_new_label_review"
        if allow_auto_add_new_label and confidence >= min_conf_allow_new_label:
            upsert_new_label_to_graph(
                db,
                domain=sample.get("domain") or infer_domain(sample) or "Unknown",
                l1=pred.get("L1", "").strip(),
                l2=pred.get("L2", "").strip(),
                l3=pred.get("L3", "").strip(),
                intent_id=sample.get("intent_id") or f"AUTO-{sample.get('sample_id', 'UNKNOWN')}",
                intent_name=pred.get("L3", "").strip() or "new_intent",
                description=pred.get("reasoning", ""),
                confidence=str(confidence),
                product_category=sample.get("category", ""),
                detection_signals=sample.get("sentence", ""),
                example=sample.get("sentence", ""),
                category_logic="auto_added_from_labeling_notebook",
            )
            graph_upserted = True
            qa_status = "approved_auto_new_label"

    ann_doc = {
        "sample_id": sample.get("sample_id"),
        "product_id": sample.get("product_id"),
        "sentence": sample.get("sentence", ""),
        "category": sample.get("category", ""),
        "model": model_name,
        "intent": {
            "level_1": pred.get("L1", "").strip(),
            "level_2": pred.get("L2", "").strip(),
            "level_3": [pred.get("L3", "").strip()] if pred.get("L3") else [],
        },
        "confidence": confidence,
        "reasoning": pred.get("reasoning", ""),
        "qa_status": qa_status,
        "new_label_pending_review": new_label_pending_review,
        "graph_upserted": graph_upserted,
        "semantic_snap_similarity": semantic_snap_similarity,
        "semantic_snap_intent_id": semantic_snap_intent_id,
        "taxonomy_semantically_snapped": used_semantic_snap,
        "source": sample.get("source", "hasaki"),
        "updated_at": now,
    }
    col_ann.update_one({"sample_id": ann_doc["sample_id"]}, {"$set": ann_doc}, upsert=True)
    return ann_doc


def _intent_slug_blob(d):
    parts = [d.get("l1"), d.get("l2"), d.get("l3"), d.get("name")]
    return " ".join(str(p or "") for p in parts).lower()


_ELECTRONICS_SLUG_MARKERS = (
    "dien_thoai",
    "laptop",
    "may_anh",
    "smartwatch",
    "tai_nghe",
    "iphone",
    "macbook",
    "gpu",
    "chuot",
)


def _candidate_looks_electronics_intent(d):
    s = _intent_slug_blob(d)
    return any(m in s for m in _ELECTRONICS_SLUG_MARKERS)


def _sentence_mentions_electronics(text):
    t = (text or "").lower()
    hints = (
        "dien thoai",
        "điện thoại",
        "laptop",
        "iphone",
        "macbook",
        "may anh",
        "máy ảnh",
        "tai nghe",
        "chuột",
        "ban phim",
        "bàn phím",
        "smartwatch",
        "card man hinh",
    )
    return any(h in t for h in hints)


def _category_match_bonus(sample, d):
    if not sample:
        return 0
    cat = (sample.get("category") or "").lower()
    pc = (d.get("product_category") or "").lower()
    if not cat or not pc:
        return 0
    cat_tokens = [w for w in re.findall(r"\w+", cat, flags=re.UNICODE) if len(w) >= 3]
    hit = sum(1 for w in cat_tokens if w in pc)
    return min(3, hit)


def get_candidate_l3_from_mongodb(db, text, category=None, top_k=10, sample=None):
    tokens = [t.lower() for t in re.findall(r"\w+", text or "", flags=re.UNICODE) if len(t) >= 2]
    if not tokens:
        return []

    or_conditions = []
    for tok in tokens:
        rgx = {"$regex": re.escape(tok), "$options": "i"}
        or_conditions.extend(
            [
                {"name": rgx},
                {"description": rgx},
                {"detection_signals": rgx},
                {"example": rgx},
                {"l3": rgx},
            ]
        )

    query = {"level": "intent", "$or": or_conditions}
    docs = list(
        db[COL_NODES].find(
            query,
            {
                "_id": 1,
                "name": 1,
                "l1": 1,
                "l2": 1,
                "l3": 1,
                "description": 1,
                "detection_signals": 1,
                "example": 1,
                "product_category": 1,
            },
        )
    )

    beauty_ctx = bool(sample) and infer_domain(sample) == "My pham"

    scored = []
    token_set = set(tokens)
    for d in docs:
        if beauty_ctx and _candidate_looks_electronics_intent(d) and not _sentence_mentions_electronics(text):
            continue

        blob = " ".join(
            [
                str(d.get("name", "")),
                str(d.get("description", "")),
                str(d.get("detection_signals", "")),
                str(d.get("example", "")),
                str(d.get("l3", "")),
                str(d.get("product_category", "")),
            ]
        ).lower()
        hit = sum(1 for t in token_set if t in blob)
        if hit <= 0:
            continue
        cat_bonus = _category_match_bonus(sample, d) if sample else 0
        scored.append((hit + 0.5 * cat_bonus, d))

    scored.sort(key=lambda x: x[0], reverse=True)
    pool = [d for _, d in scored[: max(top_k * 4, top_k + 8)]]

    seen = set()
    uniq = []
    for d in pool:
        key = (d.get("l1"), d.get("l2"), d.get("l3"))
        if key in seen:
            continue
        seen.add(key)
        uniq.append(d)
        if len(uniq) >= top_k:
            break
    return uniq


def build_retrieval_prompt(sample, candidates):
    vertical = infer_domain(sample)
    lines = []
    for i, c in enumerate(candidates, 1):
        pc = c.get("product_category") or ""
        lines.append(
            f"{i}. L1={c.get('l1')} | L2={c.get('l2')} | L3={c.get('l3')} | product_category={pc!r} | intent_id={c.get('_id')}"
        )
    taxonomy_text = (
        "\n".join(lines)
        if lines
        else "KHÔNG CÓ ỨNG VIÊN — vẫn trả JSON hợp lệ; đặt is_new_label=true nếu không khớp dòng nào."
    )

    parts = [
        "Bạn là chuyên gia gán nhãn intent cho QA thương mại điện tử (mỹ phẩm / làm đẹp và điện tử).",
        f"Ngành ước lượng từ danh mục và tên sản phẩm: **{vertical}**.",
        "QUY TẮC (tuân thủ nghiêm):",
        f"- Danh mục nằm trong catalog Hasaki (làm đẹp): {((sample.get('category') or '').strip() in get_hasaki_beauty_categories())}. Giá trị đã thấy trong prelabel: {', '.join(sorted(get_hasaki_beauty_categories()))}.",
        "- Bắt buộc đối chiếu `category` của mẫu với `product_category` trên mỗi dòng ứng viên để tránh gán nhầm ngành.",
        "- Cấm chọn ứng viên có slug điện tử (dien_thoai_*, laptop_*, may_anh_*, tai_nghe_*…) khi ngành là mỹ phẩm/làm đẹp và câu hỏi không nhắc rõ thiết bị điện tử.",
        "- Ý định chung (giá, khuyến mãi, giao hàng, cửa hàng) vẫn phải khớp ngành; không chọn nhãn kiểu \"giá điện thoại\" cho mỹ phẩm chỉ vì có từ \"giá\".",
        "- Chỉ chọn bộ L1, L2, L3 khớp đúng một dòng ứng viên (sao chép đúng chuỗi từ dòng đó).",
        "- Nếu không có dòng phù hợp: đặt is_new_label=true và giải thích rõ trong reasoning.",
        "",
        "ỨNG VIÊN (MongoDB):",
        taxonomy_text,
        "",
        "NGỮ CẢNH ĐẦU VÀO:",
        f"- Câu (sentence): {sample.get('sentence', '')}",
        f"- Danh mục (category): {sample.get('category', '')}",
        f"- Tên sản phẩm (product_name): {sample.get('product_name', '')}",
        f"- Thương hiệu (brand): {sample.get('brand', '')}",
        "",
        "Trả về DUY NHẤT một đối tượng JSON hợp lệ (KHÔNG markdown, KHÔNG text ngoài JSON).",
        "Thứ tự khóa — bắt buộc: viết reasoning trước (1–3 câu tiếng Việt), sau đó confidence và nhãn:",
        "{",
        '  "reasoning": "...",',
        '  "confidence": 0.0,',
        '  "L1": "...",',
        '  "L2": "...",',
        '  "L3": "...",',
        '  "is_new_label": false',
        "}",
    ]
    return "\n".join(parts)

_HASAKI_BEAUTY_CATEGORIES_CACHE = None  # (frozenset, mtime_ns) sau khi load thanh cong


def get_hasaki_beauty_categories():
    """Doc unique `category` tu HASAKI_JSON; cache theo mtime file (doi file -> doc lai)."""
    global _HASAKI_BEAUTY_CATEGORIES_CACHE
    path = HASAKI_JSON
    mtime = path.stat().st_mtime_ns if path.exists() else -1
    if isinstance(_HASAKI_BEAUTY_CATEGORIES_CACHE, tuple) and len(_HASAKI_BEAUTY_CATEGORIES_CACHE) == 2:
        cached, mt = _HASAKI_BEAUTY_CATEGORIES_CACHE
        if mt == mtime:
            return cached
    cats = set()
    try:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                for row in data:
                    if isinstance(row, dict):
                        c = (row.get("category") or "").strip()
                        if c:
                            cats.add(c)
    except Exception:
        pass
    if not cats:
        cats = {
            "Sức khỏe làm đẹp",
            "Trang điểm",
            "Mỹ phẩm high end",
        }
    fs = frozenset(cats)
    _HASAKI_BEAUTY_CATEGORIES_CACHE = (fs, mtime)
    return fs


def clear_hasaki_beauty_category_cache():
    """Xoa cache catalog category (sau khi doi HASAKI_JSON hoac can ep doc lai)."""
    global _HASAKI_BEAUTY_CATEGORIES_CACHE
    _HASAKI_BEAUTY_CATEGORIES_CACHE = None


def infer_domain(sample):
    cat_raw = (sample.get("category") or "").strip()
    cat = cat_raw.lower()
    name = (sample.get("product_name") or "").lower()
    blob = f"{cat} {name}"

    if cat_raw and cat_raw in get_hasaki_beauty_categories():
        return "My pham"

    electronics_kw = (
        "laptop phone dien thoai tai nghe chuot ban phim camera smartwatch "
        "gaming console ram gpu cpu may anh dong ho thong minh "
        "tablet iphone tivi tv"
    )
    if any(k in blob for k in electronics_kw.split()):
        return "Dien tu"

    cat_hints = (
        "may tinh",
        "dien tu",
        "phu kien dien",
        "dien thoai",
        "dong ho thong minh",
    )
    if any(h in cat for h in cat_hints):
        return "Dien tu"

    return "My pham"


print("Helpers OK")


Helpers OK


## 5. Tải Qwen


In [7]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="cuda",
    trust_remote_code=True,
)
print("device", next(model.parameters()).device)


RuntimeError: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

In [ ]:
!nvidia-smi

## 6. Sinh nhãn + parse JSON


In [ ]:
import json as json_lib
import re
import torch


def extract_json_object(text):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    if "```" in text:
        for chunk in text.split("```"):
            c = chunk.strip()
            if c.lower().startswith("json"):
                c = c[4:].strip()
            if c.startswith("{"):
                text = c
                break
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Khong tim thay JSON")
    return json_lib.loads(text[start : end + 1])


@torch.inference_mode()
def predict_intent(sample, top_k=None, use_semantic=True):
    """Enhanced predict_intent with graph-aware semantic retrieval and ambiguity filtering
    
    Args:
        sample: sample data dict
        top_k: max candidates (fallback to TOP_K_CANDIDATES)
        use_semantic: whether to use union retrieval (True) or regex only (False)
    
    Returns:
        tuple: (prediction, raw_output, candidates)
    """
    if top_k is None:
        top_k = TOP_K_CANDIDATES
    
    # PRE-CHECK: Skip prediction if sentence is too ambiguous
    sentence = sample.get("sentence", "").strip()
    is_ambiguous, ambiguous_reason = is_sentence_too_ambiguous_to_annotate(sentence)
    
    if is_ambiguous:
        # Return dummy prediction for ambiguous sentences
        pred = {
            "reasoning": f"Câu quá ngắn/mơ hồ để dự đoán: {ambiguous_reason}",
            "confidence": 0.0,
            "L1": "",
            "L2": "",
            "L3": "",
            "is_new_label": False,
            "_ambiguous": True,
            "_ambiguous_reason": ambiguous_reason
        }
        return pred, f"SKIPPED_AMBIGUOUS: {ambiguous_reason}", []
    
    # INTEGRATION POINT: Use union retrieval instead of regex-only
    if use_semantic:
        cands = union_retrieval(
            db, 
            sample.get("sentence", ""), 
            category=sample.get("category"),
            top_k_regex=max(6, top_k//2),
            top_k_semantic=max(4, top_k//3),
            sample=sample
        )
    else:
        # Fallback to original regex retrieval
        cands = get_candidate_l3_from_mongodb(
            db, sample.get("sentence", ""), category=sample.get("category"), top_k=top_k, sample=sample
        )
    
    user_prompt = build_retrieval_prompt(sample, cands)
    messages = [
        {"role": "system", "content": "Bạn chỉ trả về một đoạn JSON hợp lệ, không markdown, không text ngoài JSON. Tuân thủ thứ tự khóa trong yêu cầu: reasoning trước, sau đó confidence, L1, L2, L3, is_new_label."},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(gen, skip_special_tokens=True)
    pred = extract_json_object(raw)
    return pred, raw, cands


def enrich_sample(s):
    out = dict(s)
    out["domain"] = infer_domain(s)
    out["intent_id"] = f"AUTO-{s.get('sample_id', 'x')}"
    return out


## 7. Thử một mẫu


In [ ]:
# Setup Graph-Aware Semantic Embeddings (One-time)
print("🔄 Embedding intent nodes with graph-aware descriptions...")
embed_count = embed_and_store_intent_nodes(db)
print(f"✅ Embedded {embed_count} nodes")

# Test semantic retrieval
print("\n🧪 Testing semantic retrieval...")
test_query = "Sản phẩm này có bị mốc không?"
semantic_results = semantic_retrieval(db, test_query, top_k=5)

print(f"Query: '{test_query}'")
print(f"Found {len(semantic_results)} semantic matches:")
for i, result in enumerate(semantic_results, 1):
    print(f"{i}. {result.get('l1')}.{result.get('l2')}.{result.get('l3')} (score: {result['semantic_score']:.3f})")

# Test union retrieval  
print("\n🧪 Testing union retrieval...")
union_results = union_retrieval(db, test_query, top_k_regex=5, top_k_semantic=3)
print(f"Union results: {len(union_results)} candidates")
for i, result in enumerate(union_results, 1):
    method = result.get('retrieval_method', 'unknown')
    score = result.get('semantic_score', 'N/A')
    print(f"{i}. [{method}] {result.get('l1')}.{result.get('l2')}.{result.get('l3')} (score: {score})")

# Test ambiguity filtering
print("\n🧪 Testing ambiguity filtering...")

ambiguous_samples = [
    {"sample_id": "amb_1", "sentence": "Có không?", "category": "Test"},
    {"sample_id": "amb_2", "sentence": "Gì?", "category": "Test"}, 
    {"sample_id": "amb_3", "sentence": "Sao", "category": "Test"},
    {"sample_id": "amb_4", "sentence": "Bao nhiều tiền ạ?", "category": "Test"},  # Should pass
    {"sample_id": "amb_5", "sentence": "Sp tốt không", "category": "Test"},  # Should pass
]

for sample in ambiguous_samples:
    enriched = enrich_sample(sample)
    is_amb, reason = is_sentence_too_ambiguous_to_annotate(sample["sentence"])
    print(f"'{sample['sentence']}' -> Ambiguous: {is_amb} ({reason})")
    
    # Test prediction
    pred, raw, cands = predict_intent(enriched)
    if pred.get("_ambiguous"):
        print(f"   -> SKIPPED: {pred['_ambiguous_reason']}")
        # Test annotation saving
        ann = save_annotation_with_guardrails(db, enriched, pred)
        print(f"   -> Saved as: {ann['qa_status']}")
    else:
        print(f"   -> Normal processing: {len(cands)} candidates")
    print()

# Test enhanced prediction with graph-aware semantic retrieval
print("\n🧪 Testing enhanced predict_intent with union retrieval...")

sample_demo_semantic = enrich_sample({
    "sample_id": "demo_semantic",
    "product_id": "demo_002", 
    "product_name": "Kem dưỡng da",
    "brand": "CeraVe",
    "category": "Chăm sóc da",
    "sentence": "Sản phẩm này có bị hỏng do nắng nóng không?",  # Test heat damage
    "source": "demo_semantic"
})

# Compare regex-only vs union retrieval
print("\n=== REGEX-ONLY RETRIEVAL ===")
pred_regex, raw_regex, cands_regex = predict_intent(sample_demo_semantic, use_semantic=False)
print(f"Regex candidates: {len(cands_regex)}")
for i, c in enumerate(cands_regex[:3], 1):
    print(f"{i}. {c.get('l1')}.{c.get('l2')}.{c.get('l3')}")

print(f"\nPrediction: {pred_regex}")

print("\n=== UNION RETRIEVAL (REGEX + SEMANTIC) ===") 
pred_union, raw_union, cands_union = predict_intent(sample_demo_semantic, use_semantic=True)
print(f"Union candidates: {len(cands_union)}")
for i, c in enumerate(cands_union[:5], 1):
    method = c.get('retrieval_method', 'unknown')
    score = c.get('semantic_score', 'N/A') 
    print(f"{i}. [{method}] {c.get('l1')}.{c.get('l2')}.{c.get('l3')} (score: {score})")

print(f"\nPrediction: {pred_union}")

# Save annotation with union retrieval result
ann_union = save_annotation_with_guardrails(db, sample_demo_semantic, pred_union)
print(f"\nAnnotation saved - qa_status: {ann_union['qa_status']}")


## 8. Batch Hasaki với Graph-Aware Semantic Retrieval

- **Chạy full file:** cell `hasaki-full-batch-run` — `json.load` một lần, xử lý theo **`HASAKI_BATCH_SIZE`**, checkpoint `data/outputs/hasaki_ckpt_*.json`. Sau khi xong: **giải phóng** list toàn file (`del hasaki_all`) để nhường RAM.
- **Resume:** trong cell **§2 Cấu hình** đặt `HASAKI_RUN_OFFSET` = index bắt đầu (sau đó chạy lại cell full).
- **Export:** cell tiếp theo ghi `hasaki_labelled_full_n*_offset*.json`. Có thể bật `HASAKI_FREE_MEM_AFTER_EXPORT` (§2) để **xóa `BATCH_OUTPUT` khỏi RAM** sau khi ghi file (kết quả vẫn trên đĩa).
- **Cache category:** `get_hasaki_beauty_categories()` cache theo **mtime** file prelabel (bộ nhớ nhỏ); gọi `clear_hasaki_beauty_category_cache()` khi đổi file prelabel và cần ép đọc lại.
- **Semantic Retrieval:** Mặc định sử dụng union retrieval (regex + semantic) để giảm `pending_new_label_review`.


In [ ]:
def run_batch(samples):
    stats = {}
    results = []
    for s in samples:
        ex = enrich_sample(s)
        try:
            pred, raw, cands = predict_intent(ex)
            ann = save_annotation_with_guardrails(db, ex, pred)
            st = ann["qa_status"]
            stats[st] = stats.get(st, 0) + 1
            results.append(
                {
                    "sample_id": ex.get("sample_id"),
                    "sentence": ex.get("sentence"),
                    "category": ex.get("category"),
                    "prediction": pred,
                    "annotation": ann,
                    "candidate_count": len(cands),
                }
            )
            print(st, ex.get("sample_id"))
        except Exception as e:
            stats["error"] = stats.get("error", 0) + 1
            results.append(
                {
                    "sample_id": ex.get("sample_id"),
                    "sentence": ex.get("sentence"),
                    "category": ex.get("category"),
                    "error": str(e),
                }
            )
            print("ERROR", ex.get("sample_id"), e)
    return {"stats": stats, "results": results}


# --- Chay HET hasaki_prelabel ---
# CANH: HASAKI_JSON, HASAKI_BATCH_SIZE, HASAKI_RUN_OFFSET o cell "2. Cau hinh"
OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not HASAKI_JSON.exists():
    print("Khong tim thay HASAKI_JSON:", HASAKI_JSON)
elif HASAKI_BATCH_SIZE <= 0:
    raise ValueError("HASAKI_BATCH_SIZE phai > 0")
else:
    with open(HASAKI_JSON, "r", encoding="utf-8") as f:
        hasaki_all = json_lib.load(f)
    n = len(hasaki_all)
    off = int(HASAKI_RUN_OFFSET)
    bs = int(HASAKI_BATCH_SIZE)
    if off < 0 or off >= n:
        raise ValueError(f"HASAKI_RUN_OFFSET={off} khong hop le (tong mau {n})")
    print("Tong mau:", n, "| batch_size:", bs, "| offset:", off)

    agg_stats = {}
    all_results = []
    for start in range(off, n, bs):
        chunk = hasaki_all[start : start + bs]
        print("=== chunk", start, "..", start + len(chunk) - 1, f"({len(chunk)} mau)")
        out = run_batch(chunk)
        for k, v in out["stats"].items():
            agg_stats[k] = agg_stats.get(k, 0) + v
        all_results.extend(out["results"])
        ckpt = OUTPUT_DIR / f"hasaki_ckpt_{start:05d}_{start + len(chunk):05d}.json"
        with open(ckpt, "w", encoding="utf-8") as cf:
            json_lib.dump(
                {
                    "chunk_start": start,
                    "chunk_end": start + len(chunk),
                    "stats": out["stats"],
                    "results": out["results"],
                },
                cf,
                ensure_ascii=False,
                indent=2,
                default=str,
            )
        print("checkpoint:", ckpt)
        del out
        del chunk

    BATCH_OUTPUT = {"stats": agg_stats, "results": all_results}
    HASAKI_LAST_RUN = {
        "total_in_file": n,
        "labelled": len(all_results),
        "offset": off,
        "batch_size": bs,
    }
    # Giai phong list toan file khoi RAM (ket qua da gom trong BATCH_OUTPUT + tung ckpt)
    del hasaki_all
    batch = None
    import gc

    gc.collect()
    print("Xong full file. Tong stats:", agg_stats)
    print("So mau da label:", len(all_results), "| HASAKI_LAST_RUN:", HASAKI_LAST_RUN)
    print("Da xoa hasaki_all / chunk khoi RAM (cache category Hasaki van giu — nhe).")



In [ ]:
# Xuat JSON sau khi chay full hasaki_prelabel (cell "hasaki-full-batch-run")
OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH_OUTPUT" not in globals():
    print("Chua co BATCH_OUTPUT — hay chay cell full hasaki (hasaki-full-batch-run) truoc.")
elif not BATCH_OUTPUT.get("results"):
    print("BATCH_OUTPUT khong co results — kiem tra loi khi chay full file.")
else:
    nr = len(BATCH_OUTPUT["results"])
    off = int(HASAKI_RUN_OFFSET) if "HASAKI_RUN_OFFSET" in globals() else 0
    OUTPUT_FILE = OUTPUT_DIR / f"hasaki_labelled_full_n{nr}_offset{off}.json"

    payload = {
        "run_mode": "full_hasaki_prelabel",
        "hasaki_json": str(HASAKI_JSON),
        "hasaki_exists": HASAKI_JSON.exists(),
        "hasaki_batch_size": HASAKI_BATCH_SIZE,
        "hasaki_run_offset": off,
        "num_labelled": nr,
    }
    if "HASAKI_LAST_RUN" in globals():
        payload["last_run"] = HASAKI_LAST_RUN
    if "HASAKI_LAST_RUN" in globals() and isinstance(HASAKI_LAST_RUN, dict):
        payload["num_samples_in_source"] = HASAKI_LAST_RUN.get("total_in_file")
    elif "batch" in globals() and batch is not None:
        payload["num_samples_in_source"] = len(batch)

    payload["stats"] = BATCH_OUTPUT.get("stats", {})

    formatted_results = []
    for item in BATCH_OUTPUT.get("results", []):
        pred = item.get("prediction") or {}
        ann = item.get("annotation") or {}
        ann_intent = ann.get("intent") or {}

        level_1 = pred.get("L1") or ann_intent.get("level_1") or ""
        level_2 = pred.get("L2") or ann_intent.get("level_2") or ""
        level_3 = pred.get("L3")
        if not level_3:
            l3_list = ann_intent.get("level_3") or []
            level_3 = l3_list[0] if l3_list else ""

        formatted_results.append(
            {
                "sample_id": item.get("sample_id"),
                "sentence": item.get("sentence"),
                "category": item.get("category"),
                "intent_3_level": {
                    "level_1": level_1,
                    "level_2": level_2,
                    "level_3": level_3,
                },
                "confidence": ann.get("confidence", pred.get("confidence")),
                "qa_status": ann.get("qa_status"),
                "reasoning": pred.get("reasoning", ann.get("reasoning")),
                "error": item.get("error"),
            }
        )

    payload["results"] = formatted_results

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json_lib.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Da tao file:", OUTPUT_FILE)
    print("So ket qua da luu:", len(payload.get("results", [])))

    if globals().get("HASAKI_FREE_MEM_AFTER_EXPORT", False):
        globals().pop("BATCH_OUTPUT", None)
        globals().pop("batch", None)
        import gc

        gc.collect()
        print("Da giai phong BATCH_OUTPUT / batch trong RAM (HASAKI_FREE_MEM_AFTER_EXPORT=True).")



## Ghi chú & Graph-Aware Semantic Retrieval

### ✅ Đã triển khai Graph-Aware Semantic Embedding Retrieval (Level 3)

**Tính năng mới:**

1. **Ambiguity Filtering:** Lọc câu ngắn/mơ hồ không thể annotate
   - Check độ dài: ít nhất 8 ký tự có nghĩa
   - Check từ có nghĩa: ít nhất 2 từ dài ≥3 ký tự (loại stop words)
   - Check pattern chỉ có từ hỏi: "Có không?", "Gì?", "Sao?"
   - Status: `skipped_ambiguous` với reason cụ thể

2. **Graph-Aware Intent Descriptions:** Mỗi intent node được làm giàu với thông tin ngữ cảnh:
   - Full path: `L1.L2.L3` 
   - Parent category (L2)
   - Sibling intents (cùng L2, giới hạn 5)
   - Description, detection signals, examples

3. **Sentence Embedding:** Sử dụng `paraphrase-multilingual-MiniLM-L12-v2`
   - Normalize embeddings với L2 norm
   - Lưu trữ trong MongoDB: `intent_nodes.embedding` + `graph_aware_text`

4. **Semantic Retrieval:** Cosine similarity với ngưỡng ≥ 0.3
   - Input: user sentence → embedding
   - Output: top-K intents với semantic scores

5. **Union Retrieval:** Kết hợp regex + semantic
   - Deduplication by `_id`
   - Preserve original document structure
   - Mark retrieval method (`regex` | `semantic`)

6. **Integration:** Enhanced `predict_intent()`
   - Pre-check: Skip ambiguous sentences
   - `use_semantic=True` (default): union retrieval
   - `use_semantic=False`: fallback regex-only
   - Giữ nguyên Qwen prompt, guardrails, thresholds

**Constraints tuân thủ:**
- ✅ KHÔNG tự tạo intents mới (trừ existing auto-create logic)  
- ✅ KHÔNG loosened confidence thresholds
- ✅ KHÔNG thay đổi schema (chỉ thêm embedding fields + ambiguous_reason)
- ✅ Giữ nguyên helper function style
- ✅ KHÔNG refactor unrelated logic
- ✅ Ambiguous sentences: Skip prediction, save với qa_status="skipped_ambiguous"

### Chạy một lần để setup embeddings:
```python
embed_and_store_intent_nodes(db)  # One-time enrichment
```

### Force re-embed sau khi thêm intent mới:
```python
# Xóa embeddings cũ để force rebuild với intent mới
db.intent_nodes.update_many(
    {"level": "intent"},
    {"$unset": {"embedding": "", "graph_aware_text": ""}}
)
embed_and_store_intent_nodes(db)  # Re-embed với intent graph mới
```

### Index gợi ý:
```python 
db.intent_annotations.create_index("sample_id", unique=True)
db.intent_nodes.create_index([("level", 1), ("l3", 1)])
db.intent_nodes.create_index("embedding")  # For semantic search performance
```

### Auto-create label:
- `ALLOW_AUTO_ADD_NEW_LABEL=True` + conf ≥ 0.96 → upsert nhãn mới vào graph.
